# Evaluation Notebook

> **Note:** This notebook is designed to run on Google Colab with GPU.
> Running locally on CPU will be very slow.
> All outputs shown were generated on Google Colab T4 GPU.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil, zipfile

shutil.copy(
    '/content/drive/MyDrive/skin-classifier/ham10000_metadata_2026-03-14.csv',
    '/content/skin-classifier/data/raw/ham10000_metadata_2026-03-14.csv'
)

shutil.copy(
    '/content/drive/MyDrive/skin-classifier/best_model.pth',
    '/content/skin-classifier/models/best_model.pth'
)
print("CSV and model copied")

print("Extracting images...")
with zipfile.ZipFile('/content/drive/MyDrive/skin-classifier/images.zip', 'r') as z:
    z.extractall('/content/skin-classifier/data/raw/')
print(f"Done - {len(os.listdir('/content/skin-classifier/data/raw/images'))} images")

In [ ]:
%%writefile /content/skin-classifier/src/dataset.py
import os
import cv2
import torch
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.model_selection import train_test_split

IMAGE_DIR = '/content/skin-classifier/data/raw/images'
CSV_PATH  = '/content/skin-classifier/data/raw/ham10000_metadata_2026-03-14.csv'

LABEL_MAP = {
    'Nevus': 0,
    'Pigmented benign keratosis': 1,
    'Melanoma, NOS': 2,
    'Basal cell carcinoma': 3,
    'Squamous cell carcinoma, NOS': 4,
    'Dermatofibroma': 5,
    'Solar or actinic keratosis': 6
}

def get_transforms(mode='train'):
    if mode == 'train':
        return A.Compose([
            A.Resize(224, 224),
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.RandomRotate90(p=0.5),
            A.ColorJitter(brightness=0.2, contrast=0.2, p=0.5),
            A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ToTensorV2()
        ])
    else:
        return A.Compose([
            A.Resize(224, 224),
            A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ToTensorV2()
        ])

class SkinLesionDataset(Dataset):
    def __init__(self, df, image_dir, transform=None):
        self.df        = df.reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row      = self.df.iloc[idx]
        img_path = os.path.join(self.image_dir, f"{row['isic_id']}.jpg")
        image    = cv2.imread(img_path)
        image    = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        if self.transform:
            image = self.transform(image=image)['image']
        label = int(row['label'])
        return image, label

def get_dataloaders(batch_size=32):
    df = pd.read_csv(CSV_PATH)
    df = df.dropna(subset=['diagnosis_3'])
    df = df[df['diagnosis_3'].isin(LABEL_MAP.keys())].reset_index(drop=True)
    df['label'] = df['diagnosis_3'].map(LABEL_MAP)

    unique_lesions = df['lesion_id'].unique()
    train_lesions, val_lesions = train_test_split(
        unique_lesions, test_size=0.2, random_state=42
    )
    train_df = df[df['lesion_id'].isin(train_lesions)].reset_index(drop=True)
    val_df   = df[df['lesion_id'].isin(val_lesions)].reset_index(drop=True)
    print(f"Train: {len(train_df)} | Val: {len(val_df)}")

    class_counts   = train_df['label'].value_counts().to_dict()
    sample_weights = [1.0 / class_counts[l] for l in train_df['label']]
    sampler = WeightedRandomSampler(
        weights=sample_weights,
        num_samples=len(sample_weights),
        replacement=True
    )

    train_dataset = SkinLesionDataset(train_df, IMAGE_DIR, get_transforms('train'))
    val_dataset   = SkinLesionDataset(val_df,   IMAGE_DIR, get_transforms('val'))

    train_loader = DataLoader(
        train_dataset, batch_size=32,
        sampler=sampler, num_workers=2, pin_memory=True
    )
    val_loader = DataLoader(
        val_dataset, batch_size=32,
        shuffle=False, num_workers=2, pin_memory=True
    )
    return train_loader, val_loader, df

In [ ]:
%%writefile /content/skin-classifier/src/model.py
import torch
import timm

NUM_CLASSES = 7

def get_model(dropout=0.4):
    model = timm.create_model(
        'efficientnet_b3',
        pretrained=False,
        num_classes=NUM_CLASSES,
        drop_rate=dropout
    )
    return model

def get_class_weights(df):
    class_counts = df['label'].value_counts().sort_index()
    total        = len(df)
    weights      = [total / (NUM_CLASSES * count) for count in class_counts]
    return torch.tensor(weights, dtype=torch.float32)

In [ ]:
from tqdm.notebook import tqdm
from sklearn.metrics import balanced_accuracy_score

all_preds  = []
all_labels = []
all_probs  = []

with torch.no_grad():
    for images, labels in tqdm(val_loader, desc="Evaluating"):
        images  = images.to(DEVICE)
        outputs = model(images)
        probs   = torch.nn.functional.softmax(outputs, dim=1)
        preds   = outputs.argmax(dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())
        all_probs.extend(probs.cpu().numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)
all_probs  = np.array(all_probs)

print(f"Balanced Accuracy: {balanced_accuracy_score(all_labels, all_preds):.4f}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

short_names = ['Nevus', 'Pig.BK', 'Melanoma', 'BCC', 'SCC', 'Derm', 'Solar']
cm      = confusion_matrix(all_labels, all_preds)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=short_names, yticklabels=short_names, ax=axes[0])
axes[0].set_title('Confusion Matrix (counts)')
axes[0].set_ylabel('True Label')
axes[0].set_xlabel('Predicted Label')

sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=short_names, yticklabels=short_names, ax=axes[1])
axes[1].set_title('Confusion Matrix (normalized)')
axes[1].set_ylabel('True Label')
axes[1].set_xlabel('Predicted Label')

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import classification_report

LABEL_MAP_INV = {
    0: 'Nevus',
    1: 'Pigmented benign keratosis',
    2: 'Melanoma, NOS',
    3: 'Basal cell carcinoma',
    4: 'Squamous cell carcinoma, NOS',
    5: 'Dermatofibroma',
    6: 'Solar or actinic keratosis'
}

class_names = [LABEL_MAP_INV[i] for i in range(7)]
print(classification_report(all_labels, all_preds, target_names=class_names))


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

classes = ['Nevus', 'Pig.BK', 'Melanoma', 'BCC', 'SCC', 'Derm', 'Solar']
precision = [0.98, 0.42, 0.27, 0.55, 0.30, 0.26, 0.29]
recall    = [0.51, 0.59, 0.66, 0.85, 0.59, 0.92, 0.46]
f1        = [0.67, 0.49, 0.38, 0.67, 0.40, 0.41, 0.36]

x = np.arange(len(classes))
width = 0.25

fig, ax = plt.subplots(figsize=(14, 6))
ax.bar(x - width, precision, width, label='Precision', color='steelblue')
ax.bar(x,         recall,    width, label='Recall',    color='coral')
ax.bar(x + width, f1,        width, label='F1 Score',  color='green')

ax.set_xticks(x)
ax.set_xticklabels(classes)
ax.set_ylim(0, 1.1)
ax.set_ylabel('Score')
ax.set_title('Per-Class Precision, Recall and F1 Score')
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()